In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:1','cuda:6'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78941)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 56701)

# #### Device() ####
# device = cuda:4

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:4)
# edge_attr                (32464, 16)              Tensor (cuda:4)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:4)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

---

In [ ]:
from modules.layers import Dims, SetPooling, AttentionSetPooling
from modules.model import Encoder, Latent
from modules.utils import reshape

import torch
import seaborn as sns

In [3]:
dims = Dims(
    dataset=dataset,
    embed_dim=16,
    num_heads=2,
    method='set'
)

In [4]:
enc = Encoder(
    dims=dims,
    method='set',
    pooling_class=SetPooling
)

enc(_batch)['x'].shape

torch.Size([64, 305, 16])

---

In [5]:
from modules.layers import Dims, SetPooling, Sequential
from modules.utils import cloneable
import torch
import torch.nn as nn

from torch import Tensor
from typing import Literal, Union

In [6]:
@cloneable
class MultiheadSetPooling(SetPooling):
    def __init__(
        self, 
        mask:Tensor,
        num_features:int,
        dims:Dims, # for multihead
        aggregate:Literal['mean','concat']='mean',

        # lin 
        hidden_dims:list[int]=None,
        act_fn:nn.Module=None, 
        norm_fn:Literal['batch','layer']=None, 
        end_fn:Union[bool,nn.Module]=False,
        *args,
        **kwargs,
    ):
        super().__init__(mask, num_features, *args, **kwargs)

        self.aggregate = aggregate
        self.num_heads = dims.num_heads

        # layers
        self.lin = Sequential(
            in_channels=num_features,
            out_channels=self.num_sets * self.num_heads,
            layer_class=nn.Linear,
            hidden_dims=hidden_dims,
            act_fn=act_fn,
            norm_fn=norm_fn,
            end_fn=end_fn
        )

        if self.aggregate == 'concat':
            self.out_proj = Sequential(
                in_channels=self.num_heads * self.num_features,
                out_channels=num_features,
                layer_class=nn.Linear,
                hidden_dims=hidden_dims,
                act_fn=act_fn,
                norm_fn=norm_fn,
                end_fn=end_fn
            )
        else:
            self.out_proj = None

    def pool(self, x:Tensor):
        # get scores
        scores = self.lin(x) # (b, n, s * h)
        scores = scores.view(-1, self.num_nodes, self.num_heads, self.num_sets) # (b, n, h, s)
        scores = scores.permute(0, 2, 1, 3) # (b, h, n, s)

        # mask scores, get attention
        mask = self.mask # (n, s)
        scores = scores.masked_fill(mask==0, float('-inf')) # (b, h, n, s)

        # attention over nodes
        attn = torch.softmax(scores, dim=2) # (b, h, n, s)
        
        # apply attention (weighted mean) over n: n,f * n,s -> s,f
        out = torch.einsum('bnf,bhns->bhsf', x, attn) # (b, h, s, f)

        # aggregate over h: (b,h,s,f) -> (b,s,f)
        if self.aggregate == 'mean':
            out = out.mean(dim=1)

        if self.aggregate == 'concat':
            out = out.permute(0,2,1,3) # (b,s,h,f)
            out = out.reshape(-1, self.num_sets, self.num_heads * self.num_features)
            out = self.out_proj(out)

        return {'x':out, 'attn':attn}

In [7]:
enc = Encoder(
    dims=dims,
    method='set',
    pooling_class=MultiheadSetPooling,
    pooling_kwargs={'aggregate':'concat'}
)

enc(_batch)['x'].shape

torch.Size([64, 305, 16])

In [8]:
enc._orig_kwargs

{'dims': <modules.layers.Dims at 0x7f0374516630>,
 'method': 'set',
 'norm_class': None,
 'encoder_class': None,
 'pooling_class': __main__.MultiheadSetPooling,
 'hidden_dims': None,
 'act_fn': None,
 'norm_fn': None,
 'end_fn': False,
 'norm_kwargs': None,
 'encoder_kwargs': None,
 'pooling_kwargs': {'aggregate': 'concat'}}

In [9]:
from modules.train import grid
from functools import partial

In [10]:
enc_temp = partial(
    Encoder,
    dims=dims,
    method='set',
    pooling_class=MultiheadSetPooling,
)

enc_grid = grid(
    enc_temp,
    pooling_kwargs=({'aggregate':'concat'},{'aggregate':'mean'}) 
)

enc_grid.keys()

dict_keys(['poolingkwargsAggregateconcat', 'poolingkwargsAggregatemean'])

In [1]:
import torch
torch.Tensor()

tensor([])